In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
DATASETS = ["pathmnist", "histoset", "skintissue"]  # any subset of [pathmnist, histoset, skintissue]
STYLES = ["llm_short", "llm_morphology"]  # any subset of [llm_short, llm_morphology]

MODEL = "google/gemini-3.7-flash"  # OpenRouter id '<provider>/<model>', e.g. google/gemini-3.7-flash
TEMPERATURE = 0.0
SEED = 42  # any int; one seed per run
OVERWRITE = False  # True | False -- False refuses to clobber a frozen file
API_KEY = ""  # "" = fall back to Kaggle Secret OPENROUTER_API_KEY

REQUEST_INTERVAL = 1.0  # seconds between calls; 0.0 disables spacing

In [ ]:
if not API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
        print("[auth] Loaded API_KEY from Kaggle Secret 'OPENROUTER_API_KEY'.")
    except Exception as exc:
        print(f"[auth] No Kaggle Secret 'OPENROUTER_API_KEY' found ({exc}).")
        print("       Paste a key into API_KEY above, or add a Kaggle Secret named")
        print("       OPENROUTER_API_KEY (Add-ons -> Secrets) before running this cell.")
        print("       Get a key at https://openrouter.ai/keys")
else:
    print("[auth] Using API_KEY from the EDIT cell.")

assert API_KEY, "No API key available -- set API_KEY or a Kaggle Secret OPENROUTER_API_KEY"

In [ ]:
import json

import yaml

from features.descriptions import description_path, generate_descriptions

In [ ]:
assert isinstance(DATASETS, list) and DATASETS, "DATASETS must be a non-empty list"
assert isinstance(STYLES, list) and STYLES, "STYLES must be a non-empty list"
for ds in DATASETS:
    assert ds in ("pathmnist", "histoset", "skintissue"), f"unknown dataset {ds!r}"
for st in STYLES:
    assert st in ("llm_short", "llm_morphology"), f"unknown style {st!r}"

with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

print(f"model: {MODEL} | temperature: {TEMPERATURE} | seed: {SEED}")
print(f"sweep: {len(DATASETS)} datasets x {len(STYLES)} styles = "
      f"{len(DATASETS) * len(STYLES)} files")
print()
print("NOTE: a hosted model call is not bit-for-bit reproducible even at temperature=0.0.")
print("      Each written JSON file is the reproducible artifact -- not the act of")
print("      calling the API. Re-running this notebook later may produce different")
print("      text even with identical settings; that is why OVERWRITE defaults to False.")

results = []

In [ ]:
for dataset in DATASETS:
    dataset_info = config["datasets"][dataset]
    class_names = list(dataset_info["class_names"])

    for style in STYLES:
        out_path = Path(description_path(dataset, style))
        print("=" * 70)
        print(f"{dataset} / {style} -> {out_path}")

        if out_path.is_file() and not OVERWRITE:
            print(f"  SKIPPED: {out_path} already exists (this is a frozen artifact, "
                  "committed to the repo). Set OVERWRITE=True to regenerate deliberately "
                  "-- this changes the file's sha256 and invalidates every cached text "
                  "prototype built from the old text.")
            results.append((dataset, style, str(out_path), "skipped (exists)"))
            continue

        try:
            payload = generate_descriptions(
                dataset=dataset,
                style=style,
                class_names=class_names,
                model=MODEL,
                temperature=TEMPERATURE,
                seed=SEED,
                api_key=API_KEY,
                request_interval=REQUEST_INTERVAL,
            )
        except Exception as exc:
            print(f"  FAILED: {type(exc).__name__}: {exc}")
            print("  (continuing with the next pair -- re-run the notebook to retry")
            print("   just this one; the pairs already written will be skipped)")
            results.append((dataset, style, str(out_path), f"FAILED ({type(exc).__name__})"))
            continue

        assert list(payload["descriptions"]) == class_names, (
            f"{dataset}/{style}: class order drifted from config.yaml"
        )
        for name, text in payload["descriptions"].items():
            assert isinstance(text, str) and text.strip(), f"{dataset}/{style}/{name}: empty description"

        for name, text in payload["descriptions"].items():
            print(f"  [{name}] {text}")
        print(f"  sha256: {payload['sha256']}")

        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, sort_keys=False, ensure_ascii=False)
        print(f"  wrote {out_path} ({out_path.stat().st_size} bytes)")
        results.append((dataset, style, str(out_path), "written"))

print("=" * 70)
print("\nSummary:")
for dataset, style, path, status in results:
    print(f"  {dataset:12} {style:16} {status:18} {path}")

incomplete = [(d, s) for d, s, _p, status in results if status.startswith("FAILED")]
if incomplete:
    print()
    print(f"{len(incomplete)} of {len(results)} pairs did not produce a file:")
    for dataset, style in incomplete:
        print(f"    {dataset} / {style}")
    print("Re-run this notebook to retry only those (the rest are skipped as existing).")
    raise RuntimeError(f"{len(incomplete)} (dataset, style) pairs failed: {incomplete}")

In [ ]:
import shutil

DESCRIPTIONS_DIR = Path("config/descriptions")
assert DESCRIPTIONS_DIR.is_dir(), f"nothing to archive at {DESCRIPTIONS_DIR}"

WORKING = Path("/kaggle/working")
ARCHIVE = WORKING / "class_descriptions"
shutil.make_archive(str(ARCHIVE), "zip", root_dir=DESCRIPTIONS_DIR)
size_kb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e3

print(f"{ARCHIVE.name}.zip  ({size_kb:.1f} KB) contains:")
for path in sorted(DESCRIPTIONS_DIR.iterdir()):
    if path.is_file():
        print(f"    {path.name}  ({path.stat().st_size} bytes)")

print(f"""
NEXT STEPS
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. Unzip it directly into this repo's config/descriptions/ directory
     (each file inside is already named {{dataset}}_{{style}}.json, matching
     description_path() -- no renaming needed).
  3. Commit the new/changed files -- these are small text files, meant to be
     committed to git like config/prompts/, unlike every GPU-cache notebook's
     Kaggle-Dataset zip.
  4. extract_vlm_features.ipynb / run_al_main.ipynb with a matching
     DESCRIPTION_STYLE will then find them.""")